# QuantDesk Agent
An LLM agent that answers trading-desk questions by calling finance tools.

**Before running:** in Colab, open the 🔑 *Secrets* panel on the left, add `NVIDIA_API_KEY` (free at build.nvidia.com) and switch on *Notebook access*. Then *Runtime → Run all*.

In [ ]:
import os
if not os.path.exists("/content/quantdesk-agent"):
    !git clone -q https://github.com/David-Bej/quantdesk-agent.git /content/quantdesk-agent
%cd /content/quantdesk-agent
!pip install -q openai

from google.colab import userdata
os.environ["NVIDIA_API_KEY"] = userdata.get("NVIDIA_API_KEY")

## 1. The maths is tested

In [ ]:
!python -m pytest -q

## 2. The tools on their own (no LLM)
This is exactly what the LLM receives when it calls a tool.

In [ ]:
from tools import run_tool
print(run_tool("price_option", '{"ticker": "AAPL", "option_type": "call", "months": 3}'))
print(run_tool("get_portfolio_risk", '{"tickers": ["NVDA", "MSFT", "RMD"], "weights": [0.4, 0.3, 0.3]}'))
print(run_tool("get_implied_volatility", '{"ticker": "AAPL", "option_type": "call", "option_price": 20, "strike": 300, "days": 30}'))

## 3. Ask the agent

In [ ]:
from agent import ask
answer, calls = ask("What is the 1-year volatility of TSLA?")

## 4. The five test cases
Prints every tool call (name, arguments, result) and the final answer, and saves everything to `results/test_cases.md`.

In [ ]:
import run_test_cases
run_test_cases.run_all()

## 5. The tool-call limit
This question needs 15 tool calls, but only 6 are allowed. Calls 7 to 15 are refused with an error message, and the model has to answer with what it has.

In [ ]:
answer, calls = ask("Give me the 1-year volatility of all 15 stocks, one tool call per stock.")
print("\nTool calls run:", sum("limit" not in c["result"] for c in calls), "| refused:", sum("limit" in c["result"] for c in calls))

## 6. Your own question

In [ ]:
answer, calls = ask("¿Cuánto vale una put a 2 meses sobre MSFT con strike 5% por debajo del precio actual?")